In [1]:
import pandas as pd

In [2]:
# Load Production Data
prod_df = pd.read_excel("MineQuartelyDB.xlsx", header=2)

prod_df.columns = [
    "year",
    "qtr",
    "mine_id",
    "subunit",
    "mine_name",
    "quarterly_hours",
    "coal_production",
    "avg_employee_count",
]

# Keep subunits
prod_df = prod_df[[
    "year", "qtr", "mine_id", "mine_name", "subunit",
    "coal_production", "avg_employee_count"
]]

# Scrub
prod_df["coal_production"] = pd.to_numeric(
    prod_df["coal_production"], errors="coerce"
)

prod_df["avg_employee_count"] = pd.to_numeric(
    prod_df["avg_employee_count"], errors="coerce"
)

# Drop if workers are missing
prod_df = prod_df.dropna(subset=["year", "avg_employee_count"])

prod_df["year"] = prod_df["year"].astype(int)


C:\Users\mpenk\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# NOTE: Employment is computed by summing workers across subunits within each quarter,
# then averaging across reporting quarters to obtain consistent annual employment.

# Mine x quarter (collapse subunits first)
mine_qtr = (
    prod_df
    .groupby(["mine_id", "mine_name", "year", "qtr"], as_index=False)
    .agg(
        coal_production=("coal_production", "sum"),
        avg_employee_count=("avg_employee_count", "sum"),
    )
)

# Mine x year
mine_year = (
    mine_qtr
    .groupby(["mine_id", "mine_name", "year"], as_index=False)
    .agg(
        coal_output_total_year=("coal_production", "sum"),
        total_quarterly_employment=("avg_employee_count", "sum"),
        reporting_quarters=("qtr", "nunique"),
    )
)

mine_year.head()


,mine_id,mine_name,year,coal_output_total_year,total_quarterly_employment,reporting_quarters
0,103381,AUGER,2010,40052.0,15,1
1,103381,AUGER,2011,168399.0,90,4
2,103381,AUGER,2012,116534.0,96,4
3,103381,AUGER,2013,38857.0,20,4
4,103381,AUGER,2014,71758.0,21,4


In [4]:
# Annual average employment
mine_year["workers_total_mine"] = (
    mine_year["total_quarterly_employment"] /
    mine_year["reporting_quarters"]
)

mine_year["workers_total_mine"] = (
    mine_year["workers_total_mine"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

# Active mine flag
mine_year["mine_active"] = (
    mine_year["coal_output_total_year"] > 1
)

In [5]:
# Load reference

ref_df = pd.read_excel("State Mines_T.xlsx", header=2)

ref_df.columns = (
    ref_df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
)

ref_df = ref_df[["mine_id", "county", "type"]].copy()

before = len(ref_df)

ref_df = ref_df.drop_duplicates(subset=["mine_id"])

after = len(ref_df)
print(f"Duplicates dropped: {before - after}")

ref_df["type"] = (
    ref_df["type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

ref_df["county"] = (
    ref_df["county"]
    .astype(str)
    .str.strip()
    .str.title()
)

mask = ~ref_df["county"].str.endswith(" County", na=False)
ref_df.loc[mask, "county"] += " County"

ref_df = ref_df.rename(columns={"type": "mine_type"})

# Merge
mine_year = mine_year.merge(ref_df, on="mine_id", how="left")

# Drop facility to avoid inflating counts and biasing productivity
mine_year = mine_year[
    mine_year["mine_type"].isin(["surface", "underground"])
]

# Keep only producing mines (>1 ton)
mine_year = mine_year[
    mine_year["coal_output_total_year"] > 1
]

# Filter years
mine_year = mine_year[
    (mine_year["year"] >= 2000) &
    (mine_year["year"] <= 2024)
]

mcdowell_2024 = mine_year[
    (mine_year["year"] == 2024) &
    (mine_year["county"] == "Mcdowell County")
][["mine_id", "mine_name"]].drop_duplicates()

print(mcdowell_2024.sort_values("mine_name"))

C:\Users\mpenk\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Duplicates dropped: 0
       mine_id               mine_name
62174  4609606                   AUGER
7967   4602380  STRIP, QUARY, OPEN PIT
62227  4609624  STRIP, QUARY, OPEN PIT
61526  4609505  STRIP, QUARY, OPEN PIT
61419  4609484  STRIP, QUARY, OPEN PIT
61390  4609480  STRIP, QUARY, OPEN PIT
60624  4609395  STRIP, QUARY, OPEN PIT
60316  4609374  STRIP, QUARY, OPEN PIT
60432  4609386  STRIP, QUARY, OPEN PIT
62313  4609656  STRIP, QUARY, OPEN PIT
12378  4603404  STRIP, QUARY, OPEN PIT
20970  4605741  STRIP, QUARY, OPEN PIT
48370  4608647  STRIP, QUARY, OPEN PIT
61735  4609533             UNDERGROUND
61670  4609523             UNDERGROUND
61598  4609516             UNDERGROUND
61544  4609507             UNDERGROUND
57731  4609207             UNDERGROUND
34357  4607366             UNDERGROUND
61179  4609455             UNDERGROUND
48579  4608659             UNDERGROUND
62243  4609628             UNDERGROUND
55483  4609084             UNDERGROUND
58742  4609261             UNDERGROUND
575

In [6]:
count_mcdowell_2024 = (
    mine_year[
        (mine_year["year"] == 2024) &
        (mine_year["county"] == "Mcdowell County")
    ]["mine_id"]
    .nunique()
)

print(count_mcdowell_2024)

count_boone_2024 = (
    mine_year[
        (mine_year["year"] == 2024) &
        (mine_year["county"] == "Boone County")
    ]["mine_id"]
    .nunique()
)

print(count_boone_2024)

26
11


In [7]:
# County level agg
county_year = (
    mine_year
    .groupby(["county", "year"], as_index=False)
    .agg(
        mine_count=("mine_id", "nunique"),
        coal_output_total=("coal_output_total_year", "sum"),
        mine_employment=("workers_total_mine", "sum"), 
    )
)

In [8]:
check_2024 = county_year[
    (county_year["year"] == 2024) &
    (county_year["county"].isin(["Mcdowell County", "Boone County"]))
][["county", "mine_count"]]

print(check_2024)

              county  mine_count
49      Boone County          11
312  Mcdowell County          26


In [9]:
# Add fips
county_fips = pd.read_csv("county_fips.csv")

county_fips["county"] = (
    county_fips["county"]
    .astype(str)
    .str.strip()
    .str.title()
)

county_year = county_year.merge(
    county_fips,
    on="county",
    how="left"
)

# Reorder
county_year = county_year[[
    "county",
    "county_fips",
    "year",
    "mine_count",
    "coal_output_total",
    "mine_employment"
]]

In [10]:
# Final check 
print(
    mine_year.loc[
        (mine_year["year"] == 2023) &
        (mine_year["county"] == "Boone County"),
        "workers_total_mine"
    ].sum()
)

((mine_year["year"] == 2024) & 
 (mine_year["coal_output_total_year"] > 1)
).sum()

print(
    mine_year.loc[
        (mine_year["year"] == 2024) &
        (mine_year["county"] == "Boone County"),
        "mine_id"
    ].nunique()
)

county_year.head()

county_year.loc[
    county_year["year"] == 2024,
    "mine_count"
].sum()

print(
    mine_year.loc[
        mine_year["year"] == 2024,
        "workers_total_mine"
    ].sum()
)

535.0833333333333
11
11537.0


In [32]:
# Save
mine_year.to_csv("mine_year.csv", index=False)
county_year.to_csv("county_year.csv", index=False)